# Revised IFC Check Notebook
Notebook setup for comparing and validating transformed IFC/JSON outputs.

In [12]:
from pathlib import Path
import json
import pandas as pd

root = Path('..').resolve()
json_edit_dir = root / 'JSON_Edit'
revised_ifc_dir = root / 'Revised_IFC'
temp_ifc_dir = root / 'Temp_IFC'

print('Root:', root)
print('JSON_Edit exists:', json_edit_dir.exists())
print('Revised_IFC exists:', revised_ifc_dir.exists())
print('Temp_IFC exists:', temp_ifc_dir.exists())

Root: C:\Git\APS-IFC
JSON_Edit exists: True
Revised_IFC exists: True
Temp_IFC exists: True


## Review JSON Object Name, Property, and Value
Load the transformed JSON from `JSON_Edit`, flatten each object's properties, and preview the key columns for quick review.

In [13]:
import re

target_json = json_edit_dir / 'ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json'
assert target_json.exists(), f"JSON file not found: {target_json}"

elements = json.loads(target_json.read_text(encoding='utf-8'))
assert isinstance(elements, list), "Expected top-level JSON array."

rows = []
for obj in elements:
    obj_name = obj.get('Name')
    dbid = obj.get('DbId')
    props = obj.get('Properties', [])
    if isinstance(props, list):
        for prop in props:
            if not isinstance(prop, dict):
                continue
            rows.append({
                'ObjectName': obj_name,
                'DbId': dbid,
                'Property': prop.get('displayName'),
                'Value': prop.get('value')
            })

flat_df = pd.DataFrame(rows)
print(f"Loaded JSON: {target_json}")
print(f"Object count: {len(elements)}")
print(f"Property rows: {len(flat_df)}")

preview_df = flat_df[['ObjectName', 'DbId', 'Property', 'Value']].head(200).reset_index(drop=True)
with pd.option_context('display.max_rows', 220, 'display.min_rows', 220):
    display(preview_df)

Loaded JSON: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json
Object count: 180
Property rows: 3594


,ObjectName,DbId,Property,Value
0,A-1JNL9322340 - Pyramid,5,Name,A-1JNL9322340 - Pyramid
1,A-1JNL9322340 - Pyramid,5,Type,IFCBUILDINGELEMENTPROXY
2,A-1JNL9322340 - Pyramid,5,GUID,8d3567ff-612b-3878-a628-4b9d838df628
3,A-1JNL9322340 - Pyramid,5,Icon,Group
4,A-1JNL9322340 - Pyramid,5,Hidden,No
5,A-1JNL9322340 - Pyramid,5,Required,No
6,A-1JNL9322340 - Pyramid,5,Material,
7,A-1JNL9322340 - Pyramid,5,Source File,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.ifc
8,A-1JNL9322340 - Pyramid,5,GLOBALID,2DDMV$OIiuUAOeIvs3ZVOe
9,A-1JNL9322340 - Pyramid,5,NAME,A-1JNL9322340 - Pyramid


## Check remaining prefixed values
Detect values that still start with an ID-like prefix such as `1JNL9442631_...`.

In [14]:
prefix_pattern = re.compile(r'^\d[A-Za-z0-9]{7,}_.+')

name_prefixed = flat_df[
    flat_df['ObjectName'].astype(str).str.match(prefix_pattern, na=False)
].copy()

value_prefixed = flat_df[
    flat_df['Value'].astype(str).str.match(prefix_pattern, na=False)
].copy()

print(f"Prefixed ObjectName rows: {len(name_prefixed)}")
print(f"Prefixed Value rows: {len(value_prefixed)}")

if len(value_prefixed) > 0:
    print("\nExamples of remaining prefixed property values:")
    display(value_prefixed[['ObjectName', 'DbId', 'Property', 'Value']].head(50))
else:
    print("\nNo remaining prefixed property values detected.")

if len(name_prefixed) > 0:
    print("\nExamples of remaining prefixed object names:")
    display(name_prefixed[['ObjectName', 'DbId', 'Property', 'Value']].head(50))
else:
    print("\nNo remaining prefixed object names detected.")

Prefixed ObjectName rows: 0
Prefixed Value rows: 0

No remaining prefixed property values detected.

No remaining prefixed object names detected.


## Diagnostic: which JSON key matches Revised IFC best?
This is a read-only check. It does not modify JSON or IFC files; it only compares potential keys (`GUID`, `ExternalId`, `DbId`, `Name`) and reports match rates.

In [15]:
import ifcopenshell

json_path = json_edit_dir / 'ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json'
ifc_path = revised_ifc_dir / 'ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.ifc'
assert json_path.exists(), f"Missing JSON file: {json_path}"
assert ifc_path.exists(), f"Missing IFC file: {ifc_path}"

model = ifcopenshell.open(str(ifc_path))
json_data = json.loads(json_path.read_text(encoding='utf-8'))

def _guid_from_props(props):
    if not isinstance(props, list):
        return None
    for p in props:
        if not isinstance(p, dict):
            continue
        if str(p.get('displayName', '')).strip().lower() == 'guid':
            v = p.get('value')
            return str(v).strip() if isinstance(v, str) and v.strip() else None
    return None

def _ifc_external_id(entity):
    ext = getattr(entity, 'Tag', None)
    if isinstance(ext, str) and ext.strip():
        return ext.strip()
    ext2 = getattr(entity, 'ObjectType', None)
    if isinstance(ext2, str) and ext2.strip():
        return ext2.strip()
    return None

ifc_objects = model.by_type('IfcObject')
ifc_by_guid = {}
ifc_by_name = {}
ifc_by_external = {}

for e in ifc_objects:
    guid = getattr(e, 'GlobalId', None)
    name = getattr(e, 'Name', None)
    ext = _ifc_external_id(e)
    if isinstance(guid, str) and guid.strip():
        ifc_by_guid[guid.strip()] = e
    if isinstance(name, str) and name.strip():
        ifc_by_name[name.strip()] = e
    if isinstance(ext, str) and ext.strip():
        ifc_by_external[ext.strip()] = e

results = []
guid_miss_samples = []
name_miss_samples = []

for item in json_data:
    if not isinstance(item, dict):
        continue

    j_guid = _guid_from_props(item.get('Properties'))
    j_name = item.get('Name')
    j_ext = item.get('ExternalId')
    j_dbid = item.get('DbId')

    guid_hit = bool(j_guid and j_guid in ifc_by_guid)
    name_hit = bool(isinstance(j_name, str) and j_name.strip() and j_name.strip() in ifc_by_name)
    ext_hit = bool(isinstance(j_ext, str) and j_ext.strip() and j_ext.strip() in ifc_by_external)

    if not guid_hit and j_guid and len(guid_miss_samples) < 10:
        guid_miss_samples.append({'JsonGUID': j_guid, 'JsonName': j_name, 'DbId': j_dbid})
    if not name_hit and isinstance(j_name, str) and j_name.strip() and len(name_miss_samples) < 10:
        name_miss_samples.append({'JsonName': j_name, 'DbId': j_dbid, 'ExternalId': j_ext})

    results.append({
        'JsonGUID': j_guid,
        'JsonName': j_name,
        'JsonExternalId': j_ext,
        'JsonDbId': j_dbid,
        'GUID_match': guid_hit,
        'Name_match': name_hit,
        'ExternalId_like_match': ext_hit
    })

diag_df = pd.DataFrame(results)
summary = pd.DataFrame([
    {
        'Total JSON objects': len(diag_df),
        'GUID matches': int(diag_df['GUID_match'].sum()),
        'Name matches': int(diag_df['Name_match'].sum()),
        'ExternalId-like matches': int(diag_df['ExternalId_like_match'].sum())
    }
])

print('Diagnostic summary (read-only):')
display(summary)

print('\nSample rows with match flags:')
display(diag_df[['JsonName', 'JsonDbId', 'GUID_match', 'Name_match', 'ExternalId_like_match']].head(30))

print('\nSample GUID misses (if any):')
display(pd.DataFrame(guid_miss_samples))

print('\nSample Name misses (if any):')
display(pd.DataFrame(name_miss_samples))

Diagnostic summary (read-only):


,Total JSON objects,GUID matches,Name matches,ExternalId-like matches
0,180,0,2,0



Sample rows with match flags:


,JsonName,JsonDbId,GUID_match,Name_match,ExternalId_like_match
0,A-1JNL9322340 - Pyramid,5,False,False,False
1,"A-Electrical design requirements, MVS1, NER, A...",6,False,False,False
2,A-Cable Ladder 90 450,14,False,False,False
3,A-Cable Ladder 90 450,15,False,False,False
4,A-Cable Ladder 90 450,19,False,False,False
5,A-MVS Cable Ladder P1,10,False,False,False
6,A-K6 cable tray MVS P1,9,False,False,False
7,A-Cable Ladder MVS,13,False,False,False
8,A-Cable Ladder MVS,17,False,False,False
9,A-Cable Ladder MVS,18,False,False,False



Sample GUID misses (if any):


,JsonGUID,JsonName,DbId
0,8d3567ff-612b-3878-a628-4b9d838df628,A-1JNL9322340 - Pyramid,5
1,cd06ec2a-dde8-35e7-bb84-94eaba646aaa,"A-Electrical design requirements, MVS1, NER, A...",6
2,c9232a6d-75c5-3739-a3c2-3777ac67e597,A-Cable Ladder 90 450,14
3,3d931300-8be8-3a23-a464-9cedb7d98460,A-Cable Ladder 90 450,15
4,19f22aef-af24-3a1c-b63f-1703e94e50ba,A-Cable Ladder 90 450,19
5,76440f25-e447-3cc5-958e-8727a44d872d,A-MVS Cable Ladder P1,10
6,c91648b7-3272-3ce6-92e1-f4279dbf9478,A-K6 cable tray MVS P1,9
7,366c5df3-80dd-34c1-95f6-f69c55125c61,A-Cable Ladder MVS,13
8,69eed907-29dc-3444-89dd-6e9fe484ce1b,A-Cable Ladder MVS,17
9,97558d3b-c88d-3c4c-b66f-b2109faa27c0,A-Cable Ladder MVS,18



Sample Name misses (if any):


,JsonName,DbId,ExternalId
0,A-1JNL9322340 - Pyramid,5,0/0/1
1,"A-Electrical design requirements, MVS1, NER, A...",6,0/0/2
2,A-Cable Ladder 90 450,14,0/0/2/1/2
3,A-Cable Ladder 90 450,15,0/0/2/1/3
4,A-Cable Ladder 90 450,19,0/0/2/1/7
5,A-MVS Cable Ladder P1,10,0/0/2/1
6,A-K6 cable tray MVS P1,9,0/0/2/0
7,A-Cable Ladder MVS,13,0/0/2/1/1
8,A-Cable Ladder MVS,17,0/0/2/1/5
9,A-Cable Ladder MVS,18,0/0/2/1/6
